In [ ]:
# self.current_positions
ticker_list = ['AMZN', 'TSM', 'NVDA', 'META']



class RandomClass(object):
    def __init__(self, ticker_list):

        self.ticker_list = ticker_list

        self.current_positions = dict( (k,v) for k, v in [(s, 0) for s in self.ticker_list] )
    

        """
        [(S, 0) for s in self.ticker_list]

        - Creates a list of tuples. e.g. [('AAPL', 0), ('TSLA', 0), ...]
        - Each ticker has an initial position of 0

        (k, v) for k, v in [...]
        k = key
        v = value
        
        - Unpacks the list of tuples.

        dict(...)

        - Turns the sequence of (key, value) tuples into a dict

        """


port = RandomClass(ticker_list)
print(port.current_positions)











{'AAPL': 0, 'TSLA': 0, 'MSFT': 0}


In [23]:
import yfinance as yf
import os
ticker_list = ['SPY']
data_dir = '/Users/george/python-projects/ed-backtest/backtester/data/sp_constitutents'

print(ticker_list)
for t in ticker_list:
    ticker = yf.Ticker(f"{t}")
    df = ticker.history(start="2005-01-01", end="2026-01-01")

    filepath = os.path.join(data_dir, f'{t}.csv')
    df.to_csv(filepath)
    print(f"{t} data downloaded to {filepath}")
    

['SPY']


/Users/george/python-projects/ed-backtest/ed_venv/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


SPY data downloaded to /Users/george/python-projects/ed-backtest/backtester/data/sp_constitutents/SPY.csv


In [ ]:
"""
get_trade_points() WIN/LOSS TRACKING ORIGINAL LOGIC:


    if not sell_prices[i]:
        continue
    if sell_prices[i] > buy_prices[i]:
        wins += 1
        trade_pnl.append((sell_prices[i] - buy_prices[i]) * pos_sizes[i])
    elif sell_prices[i] < buy_prices[i]:
        losses += 1
        trade_pnl.append((sell_prices[i] - buy_prices[i]) * pos_sizes[i])
    elif sell_prices[i] == buy_prices[i]:
        breakeven += 1
    else:
        continue
"""

In [1]:
pnl = (186.82 - 183.26) * 136

print(pnl)

484.1600000000003


In [ ]:



def f():
    yield 1
    yield 2
    yield 3

g = f()



1

In [42]:
from queue import Queue

q = Queue()

for i in range(1, 4):
    q.put(f"Event {i}")

print(f"Queue size: {q.qsize()}")
print(f"Is empty? {q.empty()}")

while not q.empty():
    print(q.get())


Queue size: 3
Is empty? False
Event 1
Event 2
Event 3


In [ ]:

initial_fill_price = [72.46, 67.85, 152.50, 28.68]

initial_cost = sum(initial_fill_price) * 100

cash = 100000
cash -= initial_cost


final_fill_prices = [190.72, 139.88, 370.51, 248.47]
last_mkt_value = sum(final_fill_prices) * 100

final_portfolio_value = cash + last_mkt_value
print(final_portfolio_value)


# Validating portfolio final value with numbers directly from CSV, correct!


162809.0


In [ ]:
from abc import ABC, abstractmethod
from event import SignalEvent

class Strategy(ABC):

    @abstractmethod
    def calc_signals(self):

        raise NotImplementedError("calc_signal required, not implemented.")
    
class SMA_CrossoverStrategy(Strategy):
    def __init__(self, bars, events):

        self.bars = bars
        self.ticker_list = self.bars.ticker_list
        self.events = events
        self.bought, self.short = self._calc_initial_bought()

    def _calc_initial_bought(self):

        bought = {}
        for s in self.ticker_list:
            bought[s] = False

        short = {}
        for s in self.ticker_list:
            short[s] = False

        return bought, short
    

    
    def calc_signals(self, event):
        
        if event.type == "MARKET":
            for s in self.ticker_list:
                bars = self.bars.get_latest_bars(s, N=30)
                
                if bars is not None and bars != []:
                    if len(bars) >= 30:
                        
                        closes = [bar[4] for bar in bars[-30:]]
                        sma = sum(closes) / 30
                        
                        if closes[-1] > sma and self.bought[s] == False:
                            
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'LONG')
                            self.events.put(signal)
                            self.bought[s] = True

                        elif closes[-1] < sma and self.bought[s] == True:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.bought[s] = False

                        elif closes[-1] < sma and self.bought[s] == False:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'SHORT')
                            self.events.put(signal)
                            self.short[s] = True

                        elif closes[-1] > sma and self.short[s] == True:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.short[s] = False
                        
                        else:
                            continue








In [ ]:
class SMA_CrossoverStrategy(Strategy):
    def __init__(self, bars, events):

        self.bars = bars
        self.ticker_list = self.bars.ticker_list
        self.events = events
        self.bought, self.short = self._calc_initial_bought()

    def _calc_initial_bought(self):

        bought = {}
        for s in self.ticker_list:
            bought[s] = False

        short = {}
        for s in self.ticker_list:
            short[s] = False

        return bought, short
    

    
    def calc_signals(self, event):
        
        if event.type == "MARKET":
            for s in self.ticker_list:
                bars = self.bars.get_latest_bars(s, N=30)
                
                if bars is not None and bars != []:
                    if len(bars) >= 30:
                        
                        closes = [bar[4] for bar in bars[-30:]]
                        sma = sum(closes) / 30
                        
                        if closes[-1] > sma and self.bought[s] == False:
                            print(f"Close: {closes[-1]}")
                            print(f"SMA: {sma}")
                        
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'LONG')
                            self.events.put(signal)
                            self.bought[s] = True

                        elif closes[-1] < sma and self.bought[s] == True:
                            print(f"Close: {closes[-1]}")
                            print(f"SMA: {sma}")

                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.bought[s] = False

                        elif closes[-1] < sma and self.bought[s] == False:
                            print(f"Close: {closes[-1]}")
                            print(f"SMA: {sma}")
                            
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'SHORT')
                            self.events.put(signal)
                            self.short[s] = True

                        elif closes[-1] > sma and self.short[s] == True:
                            print(f"Close: {closes[-1]}")
                            print(f"SMA: {sma}")
                            
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.short[s] = False
                        
                        else:
                            continue

In [ ]:
def _open_convert_csv_files(self):
    """Opens the CSV files from the data directory, converts them into pandas DataFrames within a ticker dictionary"""
    """
    The idea is to get the data and merge them into 1 timeline.
    """

    comb_index = None
    for s in self.ticker_list:
        # Load the CSV file with no header information, indexed on date
        #print(s)
        self.ticker_data[s] = pd.read_csv(
            os.path.join(self.csv_dir, '%s.csv' % s),
            header=0, index_col=0, parse_dates=True,
            names=[ # Override the CSV column names and replace with these, for robustness and reusability.
                'datetime', 'open', 'high',
                'low', 'close', 'adj_close', 'volume'
            ]
        )
        self.ticker_data[s].sort_index(inplace=True)
        #print(self.ticker_data[s].close)
        #I've got the whole CSV of data fine.

In [ ]:
import pandas as pd
def _open_convert_csv_files(self):

    comb_index = None
    for s in self.ticker_list:

        self.ticker_data[s] = pd.read_csv(
            os.path.join(self.csv_dir, f'{s}.csv'),
            header=0,
            index_col=0,
            parse_dates=True,
            names=['datetime', 'open', 'high', 'low', 'close', 'adj_close', 'volume']
        )
        self.ticker_data[s].sort_index(inplace=True)


In [ ]:
class DualSMA_CrossoverStrategy(Strategy):
    def __init__(self, bars, events):

        self.bars = bars
        self.ticker_list = self.bars.ticker_list
        self.events = events
        self.bought, self.short = self._calc_initial_bought()
        self.short_period = 50
        self.med_period = 200

    def _calc_initial_bought(self):

        bought = {}
        for s in self.ticker_list:
            bought[s] = False

        short = {}
        for s in self.ticker_list:
            short[s] = False

        return bought, short
    

    
    def calc_signals(self, event):
        
        if event.type == "MARKET":
            for s in self.ticker_list:
                bars = self.bars.get_latest_bars(s, N=self.med_period)
                
                if bars is not None and bars != []:
                    if len(bars) >= self.med_period:
                        closes_med = [bar[4] for bar in bars]
                        closes_short = [bar[4] for bar in bars[-self.short_period:]]
                        sma_med = sum(closes_med) / self.med_period
                        sma_short = sum(closes_short) / self.short_period
                        
                        if sma_short > sma_med and self.bought[s] == False:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'LONG')
                            self.events.put(signal)
                            self.bought[s] = True

                        elif sma_short < sma_med and self.bought[s] == True:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.bought[s] = False



In [ ]:
   
"""
MEAN REVERSION STRATEGY
"""


def calc_signals(self, event):
        if event.type == "MARKET":
            bull_regimes = self.is_bull_regime()

            for s in self.ticker_list:
                #if not bull_regimes[s]:
                #    continue

                bars = self.bars.get_latest_bars(s, N=self.z_score_period)
                
                if bars is not None and bars != []:
                    if len(bars) >= self.z_score_period:
                        z_period_closes = np.array([bar[4] for bar in bars])
                        sma = np.mean(z_period_closes)
                        std = np.std(z_period_closes)
                        z_score = (z_period_closes[-1] - sma) / std

                        if z_score < -self.z_condition and not self.bought[s]:
                            print(f"GENERATING BUY SIGNAL FOR {s}: Z_Score = {z_score}")
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'LONG', use_risk_manager=True)
                            self.events.put(signal)
                            
                            self.bought[s] = True
                            self.short[s] = False
                            self.entry_bar[s] = bars[-1][1]

                        elif z_score > 2 and not self.short[s]:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'SHORT', use_risk_manager=True)
                            self.events.put(signal)
                            self.short[s] = True
                            self.bought[s] = False
                            self.entry_bar[s] = bars[-1][1]


                        elif abs(z_score) < 0.5:
                            current_bar = bars[-1][1]

                            if self.bought[s] and current_bar > self.entry_bar[s]:
                                signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                                self.events.put(signal)
                                self.bought[s] = False
                                self.entry_bar[s] = None
                                
                            if self.short[s]:
                                signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                                self.events.put(signal)
                                self.short[s] = False
                                self.entry_bar[s] = None


In [119]:

digits = [9, 9, 9]


value = 0
mult_factor = 1
digits = digits[::-1]
new_digits = []

for digit in digits:
    value += digit * mult_factor
    mult_factor = mult_factor * 10

value += 1

length = len(str(value))
for i in range(length):
    div_factor = 1 * (10 ** (length - 1))
    new_digits.append(value // div_factor)
    value -= (value // div_factor * (10 ** (length-1)))
    length -= 1

print(new_digits)






[1, 0, 0, 0]


In [136]:


def find_index_of_first_occurence(haystack, needle):
    for i in range(len(haystack) - len(needle) + 1):
        value = haystack[i: i + len(needle)]
        if value == needle:
            return i
    return -1

print(find_index_of_first_occurence(haystack="sasadbustusad", needle="sad"))


# haystack[i : i + len(needle)] : E.g. haystack[0:2] gives all values from index [0] up to and NOT including index [2].

# so it will return [0] and [1], ONLY





2


In [129]:

def length_of_last_word(s):
    s = s.split()
    return len(s[-1])


print(length_of_last_word(s="Hello World"))


def harder_length_of_last_word(s):
    count = 0
    i = len(s) - 1

    while i >= 0 and s[i] == " ":
        i-=1
    while i >= 0 and s[i] != " ":
        count += 1
        i-= 1
    return count

print(harder_length_of_last_word(s="Hello Worlds   "))







5
6


In [131]:
words = "HELLO WORLD BOY   "

count = 0
i = len(words) - 1

while i >= 0 and words[i] == " ":
    i -= 1

while i >= 0 and words[i] != " ":
    count += 1
    i -= 1


print(count)



3


In [159]:
haystack = "sadbastiosad"
needle = "sad"

for i in range(len(haystack) - len(needle) + 1):
    value = haystack[i: i + len(needle)]
    if value == needle:
        print(i)
        break
    else:
        print(-1)
        break


0


In [ ]:
# My Soltuion to Length of Longest Substring

def LengthOfLongestSubstring(s: str) -> int:
    seen = []
    count = 0
    highest_count = 0

    for i in s:
        if i not in seen:
            seen.append(i)
            count = len(seen)
            if count > highest_count:
                highest_count = count
        elif i in seen:
            pos = seen.index(i)
            del seen[0: pos+1]
            seen.append(i)
            count = len(seen)
            if count > highest_count:
                highest_count = count
    return highest_count


print(LengthOfLongestSubstring(s="bbbbbasd"))


# Each of the following commands make this solution O(n) time complexity:
#   - seen.index(i)
#   - if in seen
#   - if not in seen

# Stacked together, the worst case is that this solution is O(n**2)


# For O(1) time complexity, the soltion requires a hash map, to directly search for a value







4


In [1]:
def RLE(n: int) -> str:
    rle = "1"
    for i in range(n - 1):
        term_string = ""
        count = 1
        prev = rle[0]
        for j in rle[1:]:

            if j == prev:
                count += 1
            else:
                term_string += str(count) + prev
                count = 1
                prev = j


        term_string += str(count) + prev
        rle = term_string


    return rle

print(RLE(n=4))


KeyboardInterrupt: 

In [ ]:
ticker_list = []





Should sell GTA.
Should sell SME.


In [ ]:
from queue import Queue

pending_orders = Queue()

print(f"Initial Queue size: {pending_orders.qsize()}")

pending_orders.put("MSFT")
pending_orders.put("TSLA")
pending_orders.put("AAPL")

while True:
    order = pending_orders.get()
    print(order)
    print(pending_orders.qsize())







In [ ]:
import numpy as np

price = 194.27
quantity = 1000
direction = "BUY/SELL"

slippage = round(price * 0.0001 * np.log10(2*float(quantity)), 2)

if direction == "BUY":
    fill_price = price + slippage
else:
    fill_price = price - slippage


print(fill_price)


0.06
194.21
